In [4]:
from google.colab import files

uploaded = files.upload()

Saving Q07_baggage_handling.csv to Q07_baggage_handling (3).csv


In [5]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv("Q07_baggage_handling.csv")

In [7]:
df.head()

,baggage_tag,route_type,transfer_count,baggage_volume,route_distance_km,mishandled
0,BG00001,International,1,389,1155,No
1,BG00002,International,0,207,4814,No
2,BG00003,NaN,1,104,3034,No
3,BG00004,Domestic,2,142,7321,No
4,BG00005,Domestic,2,110,4567,No


In [8]:
df.shape

(123, 6)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   baggage_tag        123 non-null    object
 1   route_type         122 non-null    object
 2   transfer_count     123 non-null    int64 
 3   baggage_volume     123 non-null    int64 
 4   route_distance_km  123 non-null    int64 
 5   mishandled         121 non-null    object
dtypes: int64(3), object(3)
memory usage: 5.9+ KB


In [10]:
df.describe()

,transfer_count,baggage_volume,route_distance_km
count,123.000000,123.000000,123.000000
mean,1.495935,285.154472,4241.439024
std,1.096746,111.070059,2539.551991
min,0.000000,82.000000,284.000000
25%,1.000000,198.500000,1853.500000
50%,1.000000,286.000000,4405.000000
75%,3.000000,395.000000,6526.000000
max,3.000000,446.000000,8340.000000


In [11]:
df.isnull().sum()

,0
baggage_tag,0
route_type,1
transfer_count,0
baggage_volume,0
route_distance_km,0
mishandled,2


In [12]:
df[df.duplicated(keep=False)]

,baggage_tag,route_type,transfer_count,baggage_volume,route_distance_km,mishandled
60,BG00061,Domestic,1,429,8233,No
69,BG00070,Domestic,2,252,3466,No
88,BG00089,Domestic,1,362,7752,No
120,BG00061,Domestic,1,429,8233,No
121,BG00089,Domestic,1,362,7752,No
122,BG00070,Domestic,2,252,3466,No


In [13]:
df = df.drop_duplicates(subset="baggage_tag")

In [14]:
df.shape

(120, 6)

In [15]:
df.isnull().sum()

,0
baggage_tag,0
route_type,1
transfer_count,0
baggage_volume,0
route_distance_km,0
mishandled,2


In [16]:
df["route_type"] = df["route_type"].fillna(df["route_type"].mode()[0])

df["mishandled"] = df["mishandled"].fillna(df["mishandled"].mode()[0])

In [17]:
df.isnull().sum()

,0
baggage_tag,0
route_type,0
transfer_count,0
baggage_volume,0
route_distance_km,0
mishandled,0


In [18]:
df["mishandled_flag"] = (df["mishandled"] == "Yes").astype(int)

In [19]:
df.groupby("route_type")["mishandled_flag"].mean() * 100

,mishandled_flag
route_type,
Domestic,10.937500
International,10.714286


In [20]:
df.groupby("transfer_count")["mishandled_flag"].mean() * 100

,mishandled_flag
transfer_count,
0,7.692308
1,7.500000
2,9.090909
3,18.750000


In [21]:
df["complexity_score"] = (
    df["transfer_count"] * df["baggage_volume"]
)

In [22]:
df[["transfer_count",
    "baggage_volume",
    "complexity_score"]].head()

,transfer_count,baggage_volume,complexity_score
0,1,389,389
1,0,207,0
2,1,104,104
3,2,142,284
4,2,110,220


In [23]:
df["distance_group"] = pd.cut(
    df["route_distance_km"],
    [-np.inf, 3000, 6000, np.inf],
    labels=["Short", "Medium", "Long"]
)

In [24]:
df["distance_group"].value_counts()

,count
distance_group,
Medium,43
Short,42
Long,35


In [25]:
df.groupby(
    "distance_group",
    observed=True
)["mishandled_flag"].mean() * 100

,mishandled_flag
distance_group,
Short,11.904762
Medium,4.651163
Long,17.142857


In [26]:
from scipy.stats import chi2_contingency

table = pd.crosstab(
    df["route_type"],
    df["mishandled"]
)

chi2, p, dof, expected = chi2_contingency(table)

print("Chi-Square:", chi2)
print("P-value:", p)

Chi-Square: 0.0
P-value: 1.0


In [27]:
df.head()

,baggage_tag,route_type,transfer_count,baggage_volume,route_distance_km,mishandled,mishandled_flag,complexity_score,distance_group
0,BG00001,International,1,389,1155,No,0,389,Short
1,BG00002,International,0,207,4814,No,0,0,Medium
2,BG00003,Domestic,1,104,3034,No,0,104,Medium
3,BG00004,Domestic,2,142,7321,No,0,284,Long
4,BG00005,Domestic,2,110,4567,No,0,220,Medium


In [28]:
df.shape

(120, 9)

In [29]:
df.isnull().sum()

,0
baggage_tag,0
route_type,0
transfer_count,0
baggage_volume,0
route_distance_km,0
mishandled,0
mishandled_flag,0
complexity_score,0
distance_group,0


In [30]:
df.to_csv("cleaned_baggage_handling.csv", index=False)

In [31]:
from google.colab import files
files.download("cleaned_baggage_handling.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>